# 03b - Extended ML Time-Series Models

This notebook extends the experimentation from `03_ml_models.ipynb` by adding more robust Machine Learning models:
- **Random Forest**: An ensemble of decision trees, good for capturing non-linear relationships.
- **LightGBM**: Fast, distributed, high-performance gradient boosting framework.
- **CatBoost**: Gradient boosting on decision trees that handles categorical features natively.
- **Ridge Regression**: Linear regression with L2 regularization to prevent overfitting.
- **Lasso Regression**: Linear regression with L1 regularization (feature selection).

All models use recursive forecasting with lags and rolling features.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = BASE_DIR / 'artifacts'
DAILY_PATH = ARTIFACTS_DIR / 'daily_series.csv'

HORIZON = 30
N_SPLITS = 3

daily = pd.read_csv(DAILY_PATH, parse_dates=['date']).set_index('date')
print(f'XGBoost available: {XGBOOST_AVAILABLE}')
print(f'LightGBM available: {LIGHTGBM_AVAILABLE}')
print(f'CatBoost available: {CATBOOST_AVAILABLE}')
daily.head()

XGBoost available: True
LightGBM available: True
CatBoost available: False


,daily_revenue,daily_orders
date,,
2024-02-10,31740.30,122
2024-02-11,37996.92,141
2024-02-12,35297.50,139
2024-02-13,40122.00,150
2024-02-14,35806.81,152


In [2]:
def safe_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(np.abs(y_true) < 1e-9, np.nan, np.abs(y_true))
    return np.nanmean(np.abs((y_true - y_pred) / denom)) * 100

def metric_row(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': np.sqrt(mse), 'MAPE': safe_mape(y_true, y_pred)}

def build_rolling_splits(series, horizon=30, n_splits=3):
    first_train_end = len(series) - horizon * n_splits
    if first_train_end <= 35:
        raise ValueError('Not enough data for requested rolling setup.')
    out = []
    for i in range(n_splits):
        train_end = first_train_end + i * horizon
        train = series.iloc[:train_end]
        test = series.iloc[train_end:train_end + horizon]
        out.append((train, test))
    return out

def make_supervised_frame(series, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    d = pd.DataFrame({'y': series.astype(float)})
    for lag in lags:
        d[f'lag_{lag}'] = d['y'].shift(lag)
    for w in roll_windows:
        d[f'roll_mean_{w}'] = d['y'].shift(1).rolling(w).mean()
        d[f'roll_std_{w}'] = d['y'].shift(1).rolling(w).std()

    idx = d.index
    d['dow'] = idx.dayofweek
    d['is_weekend'] = (idx.dayofweek >= 5).astype(int)
    d['month'] = idx.month
    d['quarter'] = idx.quarter
    return d.dropna()

def make_feature_row(history, dt, lags=(1,2,3,7,14,28), roll_windows=(7,14)):
    row = {}
    for lag in lags:
        row[f'lag_{lag}'] = history.iloc[-lag] if len(history) >= lag else 0.0
    for w in roll_windows:
        vals = history.iloc[-w:] if len(history) >= w else history
        row[f'roll_mean_{w}'] = vals.mean() if len(vals) else 0.0
        row[f'roll_std_{w}'] = vals.std() if len(vals) > 1 else 0.0
    row['dow'] = dt.dayofweek
    row['is_weekend'] = int(dt.dayofweek >= 5)
    row['month'] = dt.month
    row['quarter'] = ((dt.month - 1) // 3) + 1
    return pd.DataFrame([row], index=[dt]).fillna(0.0)

def recursive_forecast(model, train_series, horizon):
    history = train_series.copy()
    future_idx = pd.date_range(start=history.index[-1] + pd.Timedelta(days=1), periods=horizon, freq='D')
    preds = []
    for dt in future_idx:
        x_row = make_feature_row(history, dt)
        p = float(model.predict(x_row)[0])
        p = max(0.0, p)
        preds.append(p)
        history.loc[dt] = p
    return pd.Series(preds, index=future_idx)

In [3]:
def run_ml_model(model_name, train, horizon):
    sup = make_supervised_frame(train)
    X = sup.drop(columns=['y'])
    y = sup['y']

    # Initialize model based on name
    if model_name == 'LinearRegression':
        model = LinearRegression()
    elif model_name == 'Ridge':
        model = Ridge(alpha=1.0)
    elif model_name == 'Lasso':
        model = Lasso(alpha=0.1)
    elif model_name == 'RandomForest':
        model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
    elif model_name == 'XGBoost':
        if not XGBOOST_AVAILABLE: raise ImportError("XGBoost not installed")
        model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42)
    elif model_name == 'LightGBM':
        if not LIGHTGBM_AVAILABLE: raise ImportError("LightGBM not installed")
        model = LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1)
    elif model_name == 'CatBoost':
        if not CATBOOST_AVAILABLE: raise ImportError("CatBoost not installed")
        model = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=4, random_state=42, verbose=0)
    else:
        raise ValueError(f"Unknown model: {model_name}")

    model.fit(X, y)
    pred = recursive_forecast(model, train, horizon)
    return pred

In [4]:
targets = ['daily_revenue', 'daily_orders']
base_models = ['LinearRegression', 'Ridge', 'Lasso', 'RandomForest']
if XGBOOST_AVAILABLE: base_models.append('XGBoost')
if LIGHTGBM_AVAILABLE: base_models.append('LightGBM')
if CATBOOST_AVAILABLE: base_models.append('CatBoost')

extended_rows = []
for target in targets:
    print(f'\nRunning evaluation for: {target}')
    series = daily[target].astype(float)
    splits = build_rolling_splits(series, horizon=HORIZON, n_splits=N_SPLITS)
    
    for split_no, (train, test) in enumerate(splits, start=1):
        for model_name in base_models:
            try:
                pred = run_ml_model(model_name, train, len(test))
                pred = pd.Series(pred.values, index=test.index)
                m = metric_row(test.values, pred.values)
                extended_rows.append({
                    'target': target,
                    'split': split_no,
                    'model': model_name,
                    **m
                })
                print(f'  [Split {split_no}] {model_name}: RMSE={m["RMSE"]:.2f}')
            except Exception as ex:
                print(f'  [Split {split_no}] {model_name} Error: {ex}')

ml_extended_cv = pd.DataFrame(extended_rows)
ml_extended_summary = (
    ml_extended_cv
    .groupby(['target', 'model'], as_index=False)[['MAE', 'MSE', 'RMSE', 'MAPE']]
    .mean()
    .sort_values(['target', 'RMSE'])
)

ml_extended_cv.to_csv(ARTIFACTS_DIR / 'ml_extended_cv_metrics.csv', index=False)
ml_extended_summary.to_csv(ARTIFACTS_DIR / 'ml_extended_metrics_summary.csv', index=False)
ml_extended_summary


Running evaluation for: daily_revenue
  [Split 1] LinearRegression: RMSE=3805.16
  [Split 1] Ridge: RMSE=3803.72
  [Split 1] Lasso: RMSE=3805.01
  [Split 1] RandomForest: RMSE=3686.41
  [Split 1] XGBoost: RMSE=4648.72
  [Split 1] LightGBM: RMSE=4666.86
  [Split 2] LinearRegression: RMSE=2950.40
  [Split 2] Ridge: RMSE=2949.39
  [Split 2] Lasso: RMSE=2950.37
  [Split 2] RandomForest: RMSE=2954.34
  [Split 2] XGBoost: RMSE=3870.96
  [Split 2] LightGBM: RMSE=2985.17
  [Split 3] LinearRegression: RMSE=3592.11
  [Split 3] Ridge: RMSE=3592.87
  [Split 3] Lasso: RMSE=3592.22
  [Split 3] RandomForest: RMSE=3711.92
  [Split 3] XGBoost: RMSE=3777.65
  [Split 3] LightGBM: RMSE=3439.56

Running evaluation for: daily_orders
  [Split 1] LinearRegression: RMSE=15.10
  [Split 1] Ridge: RMSE=15.09
  [Split 1] Lasso: RMSE=14.93
  [Split 1] RandomForest: RMSE=14.59
  [Split 1] XGBoost: RMSE=18.36
  [Split 1] LightGBM: RMSE=15.82
  [Split 2] LinearRegression: RMSE=10.61
  [Split 2] Ridge: RMSE=10.61
  [S

,target,model,MAE,MSE,RMSE,MAPE
0,daily_orders,Lasso,10.098949,1.615055e+02,12.580005,7.626157
4,daily_orders,Ridge,10.127362,1.633925e+02,12.647709,7.654068
2,daily_orders,LinearRegression,10.127928,1.634950e+02,12.651407,7.654634
3,daily_orders,RandomForest,10.377590,1.710715e+02,12.998611,7.893480
1,daily_orders,LightGBM,10.355825,1.768774e+02,13.158894,7.737705
5,daily_orders,XGBoost,11.973191,2.415528e+02,15.393428,9.091281
10,daily_revenue,Ridge,2833.305012,1.202532e+07,3448.662370,8.434346
6,daily_revenue,Lasso,2833.976078,1.202894e+07,3449.199552,8.436739
8,daily_revenue,LinearRegression,2833.988901,1.202910e+07,3449.220633,8.436801
9,daily_revenue,RandomForest,2853.838211,1.203203e+07,3450.889926,8.483187
